# Rajasthan Crop Production Data Collection

This notebook collects and inspects district-wise crop-production data for the
Rajasthan Crop Recommendation project.

The official Government of India OGD API was attempted first, but both the
resource and catalog endpoints timed out. Therefore, a Kaggle mirror of the
government crop-production dataset is used for development.

Official source:
https://www.data.gov.in/resource/district-wise-season-wise-crop-production-statistics-1997

Dataset mirror:
https://www.kaggle.com/datasets/nikhilmahajan29/crop-production-statistics-india

In [10]:
from pathlib import Path
import shutil
import kagglehub

# Find the project root.
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

# Locate the Kaggle dataset.
dataset_folder = Path(
    kagglehub.dataset_download(
        "nikhilmahajan29/crop-production-statistics-india"
    )
)

source_file = dataset_folder / "APY.csv"

# Create the project destination.
raw_data_folder = project_root / "data" / "raw"
raw_data_folder.mkdir(parents=True, exist_ok=True)

csv_path = raw_data_folder / "APY.csv"

# Copy only when the project copy does not already exist.
if not csv_path.exists():
    shutil.copy2(source_file, csv_path)
    print("APY.csv was copied into the project.")
else:
    print("APY.csv is already present in the project.")

print("File exists:", csv_path.exists())
print("File location:", csv_path)
print("File size:", csv_path.stat().st_size, "bytes")

APY.csv was copied into the project.
File exists: True
File location: c:\Users\lalit\OneDrive\Desktop\rajasthan-crop-recommender\data\raw\APY.csv
File size: 21143323 bytes


## Load and inspect the crop-production dataset

In [11]:
import pandas as pd

crop_data = pd.read_csv(csv_path)

print("Dataset shape:", crop_data.shape)

print("\nColumn names:")
print(crop_data.columns.tolist())

print("\nData types:")
print(crop_data.dtypes)

print("\nFirst five rows:")
display(crop_data.head())

Dataset shape: (345336, 8)

Column names:
['State', 'District ', 'Crop', 'Crop_Year', 'Season', 'Area ', 'Production', 'Yield']

Data types:
State             str
District          str
Crop              str
Crop_Year       int64
Season            str
Area          float64
Production    float64
Yield         float64
dtype: object

First five rows:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
0,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Kharif,2439.6,3415.0,1.40
1,Andaman and Nicobar Island,NICOBARS,Arecanut,2007,Rabi,1626.4,2277.0,1.40
2,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Autumn,4147.0,3060.0,0.74
3,Andaman and Nicobar Island,NICOBARS,Arecanut,2008,Summer,4147.0,2660.0,0.64
4,Andaman and Nicobar Island,NICOBARS,Arecanut,2009,Autumn,4153.0,3120.0,0.75


## Filter Rajasthan records

In [12]:
# Find the state column without assuming its exact spelling.
state_column = next(
    column
    for column in crop_data.columns
    if column.strip().lower() in {"state", "state_name"}
)

# Remove extra spaces and compare state names without case sensitivity.
state_names = crop_data[state_column].astype(str).str.strip()

rajasthan_data = crop_data[
    state_names.str.casefold() == "rajasthan"
].copy()

print("State column:", state_column)
print("Rows in complete dataset:", len(crop_data))
print("Rows for Rajasthan:", len(rajasthan_data))

display(rajasthan_data.head())

State column: State
Rows in complete dataset: 345336
Rows for Rajasthan: 20363


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
234004,Rajasthan,AJMER,Arhar/Tur,1998,Kharif,3.0,4.0,1.33
234005,Rajasthan,AJMER,Arhar/Tur,1999,Kharif,3.0,1.0,0.33
234006,Rajasthan,AJMER,Arhar/Tur,2000,Kharif,43.0,NaN,0.00
234007,Rajasthan,AJMER,Arhar/Tur,2001,Kharif,4.0,2.0,0.50
234008,Rajasthan,AJMER,Arhar/Tur,2002,Kharif,36.0,NaN,0.00


## Profile the Rajasthan subset


In [14]:
# Detect important columns without assuming their exact spelling.
district_column = next(
    column
    for column in rajasthan_data.columns
    if "district" in column.strip().lower()
)

year_column = next(
    column
    for column in rajasthan_data.columns
    if "year" in column.strip().lower()
)

season_column = next(
    column
    for column in rajasthan_data.columns
    if "season" in column.strip().lower()
)

crop_column = next(
    column
    for column in rajasthan_data.columns
    if column.strip().lower() in {"crop", "crop_name"}
)

print("Detected district column:", repr(district_column))
print("Detected year column:", repr(year_column))
print("Detected season column:", repr(season_column))
print("Detected crop column:", repr(crop_column))

print("\nRajasthan dataset shape:", rajasthan_data.shape)

print("\nNumber of districts:")
print(rajasthan_data[district_column].nunique())

print("\nDistrict names:")
print(sorted(rajasthan_data[district_column].dropna().unique()))

print("\nYear range:")
print(
    rajasthan_data[year_column].min(),
    "to",
    rajasthan_data[year_column].max()
)

print("\nSeasons:")
print(
    sorted(
        rajasthan_data[season_column]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )
)

print("\nNumber of different crops:")
print(rajasthan_data[crop_column].nunique())

print("\nMissing values:")
print(rajasthan_data.isna().sum())

print("\nExact duplicate rows:")
print(rajasthan_data.duplicated().sum())

Detected district column: 'District '
Detected year column: 'Crop_Year'
Detected season column: 'Season'
Detected crop column: 'Crop'

Rajasthan dataset shape: (20363, 8)

Number of districts:
33

District names:
['AJMER', 'ALWAR', 'BANSWARA', 'BARAN', 'BARMER', 'BHARATPUR', 'BHILWARA', 'BIKANER', 'BUNDI', 'CHITTORGARH', 'CHURU', 'DAUSA', 'DHOLPUR', 'DUNGARPUR', 'GANGANAGAR', 'HANUMANGARH', 'JAIPUR', 'JAISALMER', 'JALORE', 'JHALAWAR', 'JHUNJHUNU', 'JODHPUR', 'KARAULI', 'KOTA', 'NAGAUR', 'PALI', 'PRATAPGARH', 'RAJSAMAND', 'SAWAI MADHOPUR', 'SIKAR', 'SIROHI', 'TONK', 'UDAIPUR']

Year range:
1997 to 2019

Seasons:
['Kharif', 'Rabi', 'Whole Year']

Number of different crops:
40

Missing values:
State           0
District        0
Crop            0
Crop_Year       0
Season          0
Area            0
Production    620
Yield           0
dtype: int64

Exact duplicate rows:
0


In [15]:
# Work on a separate copy and preserve rajasthan_data unchanged.
rajasthan_clean = rajasthan_data.copy()

# Remove spaces from the beginning and end of every column name.
rajasthan_clean.columns = rajasthan_clean.columns.str.strip()

# Remove unwanted spaces from text values.
text_columns = ["State", "District", "Season", "Crop"]

for column in text_columns:
    rajasthan_clean[column] = (
        rajasthan_clean[column]
        .astype(str)
        .str.strip()
    )

print("Cleaned column names:")
print(rajasthan_clean.columns.tolist())

print("\nNon-zero missing-value counts:")
missing_counts = rajasthan_clean.isna().sum()
print(missing_counts[missing_counts > 0])

print("\nDuplicates after text cleaning:")
print(rajasthan_clean.duplicated().sum())

display(rajasthan_clean.head())

Cleaned column names:
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield']

Non-zero missing-value counts:
Production    620
dtype: int64

Duplicates after text cleaning:
0


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
234004,Rajasthan,AJMER,Arhar/Tur,1998,Kharif,3.0,4.0,1.33
234005,Rajasthan,AJMER,Arhar/Tur,1999,Kharif,3.0,1.0,0.33
234006,Rajasthan,AJMER,Arhar/Tur,2000,Kharif,43.0,NaN,0.00
234007,Rajasthan,AJMER,Arhar/Tur,2001,Kharif,4.0,2.0,0.50
234008,Rajasthan,AJMER,Arhar/Tur,2002,Kharif,36.0,NaN,0.00


## Audit missing and unusual numeric values


In [16]:
numeric_columns = ["Area", "Production", "Yield"]

print("Numeric summary:")
display(rajasthan_clean[numeric_columns].describe())

print("\nRows with missing production:")
missing_production = rajasthan_clean[
    rajasthan_clean["Production"].isna()
]

print("Count:", len(missing_production))
print(
    "Percentage:",
    round(len(missing_production) / len(rajasthan_clean) * 100, 2),
    "%"
)

print("\nYield values where production is missing:")
print(missing_production["Yield"].value_counts(dropna=False))

print("\nZero-value counts:")
print((rajasthan_clean[numeric_columns] == 0).sum())

print("\nNegative-value counts:")
print((rajasthan_clean[numeric_columns] < 0).sum())

print("\nExample rows with missing production:")
display(missing_production.head(10))

Numeric summary:


,Area,Production,Yield
count,2.036300e+04,1.974300e+04,20363.000000
mean,2.388908e+04,2.984168e+04,3.312474
std,6.958621e+04,8.841677e+04,13.032894
min,1.000000e+00,0.000000e+00,0.000000
25%,3.200000e+01,4.000000e+01,0.500000
50%,4.990000e+02,6.370000e+02,1.000000
75%,1.109250e+04,1.229750e+04,1.900000
max,1.133397e+06,1.329576e+06,1022.000000



Rows with missing production:
Count: 620
Percentage: 3.04 %

Yield values where production is missing:
Yield
0.0    620
Name: count, dtype: int64

Zero-value counts:
Area            0
Production     81
Yield         721
dtype: int64

Negative-value counts:
Area          0
Production    0
Yield         0
dtype: int64

Example rows with missing production:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
234006,Rajasthan,AJMER,Arhar/Tur,2000,Kharif,43.0,NaN,0.0
234008,Rajasthan,AJMER,Arhar/Tur,2002,Kharif,36.0,NaN,0.0
234010,Rajasthan,AJMER,Arhar/Tur,2004,Kharif,3.0,NaN,0.0
234012,Rajasthan,AJMER,Arhar/Tur,2009,Kharif,6.0,NaN,0.0
234016,Rajasthan,AJMER,Arhar/Tur,2016,Kharif,2.0,NaN,0.0
234079,Rajasthan,BARAN,Arhar/Tur,2014,Kharif,3.0,NaN,0.0
234084,Rajasthan,BARMER,Arhar/Tur,2000,Kharif,3.0,NaN,0.0
234110,Rajasthan,BHILWARA,Arhar/Tur,2000,Kharif,2.0,NaN,0.0
234112,Rajasthan,BHILWARA,Arhar/Tur,2002,Kharif,5.0,NaN,0.0
234114,Rajasthan,BHILWARA,Arhar/Tur,2005,Kharif,4.0,NaN,0.0


In [17]:
rows_before = len(rajasthan_clean)

model_data = (
    rajasthan_clean
    .dropna(subset=["Production"])
    .copy()
)

rows_after = len(model_data)
rows_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Rows removed:", rows_removed)
print("Rows remaining:", rows_after)

print("\nMissing values after removal:")
print(model_data.isna().sum())

Rows before: 20363
Rows removed: 620
Rows remaining: 19743

Missing values after removal:
State         0
District      0
Crop          0
Crop_Year     0
Season        0
Area          0
Production    0
Yield         0
dtype: int64


## Validate area, production and yield

In [18]:
import numpy as np

numeric_columns = ["Area", "Production", "Yield"]

print("Zero-value counts:")
print((model_data[numeric_columns] == 0).sum())

print("\nNegative-value counts:")
print((model_data[numeric_columns] < 0).sum())

print("\nRows with area less than or equal to zero:")
print((model_data["Area"] <= 0).sum())

# Calculate yield independently.
calculated_yield = model_data["Production"] / model_data["Area"]

# Compare calculated yield with the supplied yield.
yield_matches = np.isclose(
    model_data["Yield"],
    calculated_yield,
    atol=0.01,
    rtol=0
)

print("\nRows where yield matches Production / Area:")
print(yield_matches.sum())

print("\nRows where yield does not match:")
print((~yield_matches).sum())

# Examine questionable records.
questionable_rows = model_data[
    (model_data["Area"] <= 0)
    | (model_data["Production"] <= 0)
    | (model_data["Yield"] <= 0)
    | (~yield_matches)
]

print("\nQuestionable rows:", len(questionable_rows))
display(questionable_rows.head(20))

Zero-value counts:
Area            0
Production     81
Yield         101
dtype: int64

Negative-value counts:
Area          0
Production    0
Yield         0
dtype: int64

Rows with area less than or equal to zero:
0

Rows where yield matches Production / Area:
19669

Rows where yield does not match:
74

Questionable rows: 179


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
234355,Rajasthan,JHUNJHUNU,Arhar/Tur,2011,Kharif,2.0,0.0,0.00
234721,Rajasthan,BIKANER,Bajra,1999,Kharif,131853.0,0.0,0.00
235253,Rajasthan,SIROHI,Bajra,2017,Kharif,16610.0,14.0,0.00
235304,Rajasthan,ALWAR,Banana,2002,Whole Year,1.0,0.0,0.00
235305,Rajasthan,ALWAR,Banana,2003,Whole Year,2.0,0.0,0.00
235315,Rajasthan,BANSWARA,Banana,2003,Whole Year,2.0,0.0,0.00
235361,Rajasthan,PALI,Banana,2002,Whole Year,2.0,0.0,0.00
235362,Rajasthan,PALI,Banana,2003,Whole Year,1.0,0.0,0.00
235370,Rajasthan,RAJSAMAND,Banana,2002,Whole Year,1.0,0.0,0.00
235371,Rajasthan,RAJSAMAND,Banana,2003,Whole Year,3.0,0.0,0.00


## Investigate zero values and yield differences

In [19]:
# Create a temporary calculated yield for investigation
validation_data = model_data.copy()

validation_data["Calculated_Yield"] = (
    validation_data["Production"] / validation_data["Area"]
)

validation_data["Yield_Difference"] = abs(
    validation_data["Yield"] - validation_data["Calculated_Yield"]
)

print("Production = 0 and Yield = 0:")
print(
    len(
        validation_data[
            (validation_data["Production"] == 0)
            & (validation_data["Yield"] == 0)
        ]
    )
)

print("\nProduction > 0 but Yield = 0:")
print(
    len(
        validation_data[
            (validation_data["Production"] > 0)
            & (validation_data["Yield"] == 0)
        ]
    )
)

print("\nProduction = 0 but Yield > 0:")
print(
    len(
        validation_data[
            (validation_data["Production"] == 0)
            & (validation_data["Yield"] > 0)
        ]
    )
)

print("\nYield-difference summary:")
display(validation_data["Yield_Difference"].describe())

print("\nRows with the largest yield differences:")
display(
    validation_data.sort_values(
        "Yield_Difference",
        ascending=False
    ).head(20)
)

Production = 0 and Yield = 0:
70

Production > 0 but Yield = 0:
31

Production = 0 but Yield > 0:
11

Yield-difference summary:


count    19743.000000
mean         0.002307
std          0.003919
min          0.000000
25%          0.000539
50%          0.002110
75%          0.003488
max          0.200000
Name: Yield_Difference, dtype: float64


Rows with the largest yield differences:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield,Calculated_Yield,Yield_Difference
237762,Rajasthan,JHALAWAR,Cotton(lint),2018,Kharif,1.0,6.0,5.80,6.0,0.20
237582,Rajasthan,DAUSA,Cotton(lint),2013,Kharif,1.0,6.0,5.88,6.0,0.12
237834,Rajasthan,KOTA,Cotton(lint),2014,Kharif,1.0,6.0,5.88,6.0,0.12
237759,Rajasthan,JHALAWAR,Cotton(lint),2014,Kharif,1.0,6.0,5.88,6.0,0.12
237410,Rajasthan,BARAN,Cotton(lint),2013,Kharif,1.0,6.0,5.88,6.0,0.12
237930,Rajasthan,SAWAI MADHOPUR,Cotton(lint),2015,Kharif,1.0,6.0,5.88,6.0,0.12
250838,Rajasthan,ALWAR,Small millets,2002,Kharif,1.0,0.0,0.10,0.0,0.10
237428,Rajasthan,BARMER,Cotton(lint),2014,Kharif,3.0,12.0,3.92,4.0,0.08
237427,Rajasthan,BARMER,Cotton(lint),2013,Kharif,3.0,12.0,3.92,4.0,0.08
237758,Rajasthan,JHALAWAR,Cotton(lint),2012,Kharif,3.0,12.0,3.92,4.0,0.08


## Recalculate yield from production and area

In [20]:
# Keep a separate final-cleaning DataFrame
clean_data = model_data.copy()

# Recalculate Yield using its mathematical definition
clean_data["Yield"] = (
    clean_data["Production"] / clean_data["Area"]
)

print("Dataset shape:", clean_data.shape)

print("\nMissing values:")
print(clean_data.isnull().sum())

print("\nNegative values:")
print((clean_data[["Area", "Production", "Yield"]] < 0).sum())

print("\nZero values after recalculating Yield:")
print((clean_data[["Area", "Production", "Yield"]] == 0).sum())

print("\nFirst five rows:")
display(clean_data.head())

Dataset shape: (19743, 8)

Missing values:
State         0
District      0
Crop          0
Crop_Year     0
Season        0
Area          0
Production    0
Yield         0
dtype: int64

Negative values:
Area          0
Production    0
Yield         0
dtype: int64

Zero values after recalculating Yield:
Area           0
Production    81
Yield         81
dtype: int64

First five rows:


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
234004,Rajasthan,AJMER,Arhar/Tur,1998,Kharif,3.0,4.0,1.333333
234005,Rajasthan,AJMER,Arhar/Tur,1999,Kharif,3.0,1.0,0.333333
234007,Rajasthan,AJMER,Arhar/Tur,2001,Kharif,4.0,2.0,0.500000
234009,Rajasthan,AJMER,Arhar/Tur,2003,Kharif,15.0,15.0,1.000000
234011,Rajasthan,AJMER,Arhar/Tur,2007,Kharif,57.0,12.0,0.210526


## Save the cleaned Rajasthan dataset

In [21]:
from pathlib import Path
import pandas as pd

# Find the project root whether the notebook is running
# from the project folder or the notebooks folder
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

output_path = (
    project_root
    / "data"
    / "processed"
    / "rajasthan_crop_production_clean.csv"
)

# Ensure that the processed folder exists
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save without pandas' extra index column
clean_data.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("Location:", output_path)
print("File exists:", output_path.exists())
print("File size:", output_path.stat().st_size, "bytes")

Cleaned dataset saved successfully.
Location: c:\Users\lalit\OneDrive\Desktop\rajasthan-crop-recommender\data\processed\rajasthan_crop_production_clean.csv
File exists: True
File size: 1392170 bytes


In [22]:
saved_data = pd.read_csv(output_path)

print("Saved dataset shape:", saved_data.shape)
print("Expected shape:", clean_data.shape)

print("\nColumns:")
print(saved_data.columns.tolist())

print("\nTotal missing values:")
print(saved_data.isnull().sum().sum())

print("\nExact duplicate rows:")
print(saved_data.duplicated().sum())

print("\nData types:")
print(saved_data.dtypes)

display(saved_data.head())

Saved dataset shape: (19743, 8)
Expected shape: (19743, 8)

Columns:
['State', 'District', 'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Yield']

Total missing values:
0

Exact duplicate rows:
0

Data types:
State             str
District          str
Crop              str
Crop_Year       int64
Season            str
Area          float64
Production    float64
Yield         float64
dtype: object


,State,District,Crop,Crop_Year,Season,Area,Production,Yield
0,Rajasthan,AJMER,Arhar/Tur,1998,Kharif,3.0,4.0,1.333333
1,Rajasthan,AJMER,Arhar/Tur,1999,Kharif,3.0,1.0,0.333333
2,Rajasthan,AJMER,Arhar/Tur,2001,Kharif,4.0,2.0,0.500000
3,Rajasthan,AJMER,Arhar/Tur,2003,Kharif,15.0,15.0,1.000000
4,Rajasthan,AJMER,Arhar/Tur,2007,Kharif,57.0,12.0,0.210526
